In [1]:
!pip -q install pyspark pytest openpyxl

In [2]:
from google.colab import files
uploaded = files.upload();

Saving Products.csv to Products.csv
Saving Orders.json to Orders.json
Saving Customer.xlsx to Customer.xlsx


In [3]:
import glob,json
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").getOrCreate()



In [4]:
c = pd.read_excel(glob.glob("/content/Customer*.xlsx")[0], dtype=str).fillna("")
raw_customers = spark.createDataFrame(c)

In [ ]:
raw_customers.show(5)

In [5]:
p = pd.read_csv(glob.glob("/content/Products*.csv")[0])
raw_products = spark.createDataFrame(p)

In [ ]:
raw_products.show(5)

In [6]:
o = pd.DataFrame(json.load(open(glob.glob("/content/Orders*.json")[0])))
raw_orders = spark.createDataFrame(o)

In [ ]:
raw_orders.show(2)

Create raw tables for each source dataset

In [7]:
raw_customers.show(3, truncate=False)
raw_products.show(3, truncate=False)
raw_orders.show(3, truncate=False)

+-----------+--------------+---------------------------+--------------------+-------------------------------------------------+-----------+-------------+----------------+--------+-----------+-------+
|Customer ID|Customer Name |email                      |phone               |address                                          |Segment    |Country      |City            |State   |Postal Code|Region |
+-----------+--------------+---------------------------+--------------------+-------------------------------------------------+-----------+-------------+----------------+--------+-----------+-------+
|PW-19240   |Pierre Wener  |bettysullivan808@gmail.com |421.580.0902x9815   |001 Jones Ridges Suite 338\nJohnsonfort, FL 95462|Consumer   |United States|Louisville      |Colorado|80027      |West   |
|GH-14410   |Gary567 Hansen|austindyer948@gmail.com    |001-542-415-0246x314|00347 Murphy Unions\nAshleyton, IA 29814         |Home Office|United States|Chicago         |Illinois|60653      |Central|


In [8]:
print(raw_customers.count(), raw_products.count(), raw_orders.count())

793 1851 9994


Create an enriched table for customers and products


In [9]:
enriched_customers = raw_customers.select(
    "Customer ID", "Customer Name", "Segment", "Country", "City", "State", "Region")

enriched_products = (raw_products
    .dropDuplicates(["Product ID"])
    .select("Product ID", "Category", "Sub-Category", "Product Name"))



In [10]:
print(enriched_customers.count())
print(enriched_products.count())

793
1818


In [11]:
raw_orders.show(2)

+------+--------------+----------+---------+--------------+-----------+---------------+--------+------+--------+------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|     Product ID|Quantity| Price|Discount|Profit|
+------+--------------+----------+---------+--------------+-----------+---------------+--------+------+--------+------+
|     1|CA-2016-122581| 21/8/2016|25/8/2016|Standard Class|   JK-15370|FUR-CH-10002961|       7|573.17|     0.3| 63.69|
|     2|CA-2017-117485| 23/9/2017|29/9/2017|Standard Class|   BD-11320|TEC-AC-10004659|       4|291.96|     0.0|102.19|
+------+--------------+----------+---------+--------------+-----------+---------------+--------+------+--------+------+
only showing top 2 rows


In [12]:
orders = raw_orders.withColumn("Profit", F.round("Profit", 2))

In [13]:
orders = orders.join(
    enriched_customers.select("Customer ID", "Customer Name", "Country"),
    on="Customer ID", how="left")

In [14]:
orders.show(2)

+-----------+------+--------------+----------+---------+--------------+---------------+--------+------+--------+------+--------------+-------------+
|Customer ID|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|     Product ID|Quantity| Price|Discount|Profit| Customer Name|      Country|
+-----------+------+--------------+----------+---------+--------------+---------------+--------+------+--------+------+--------------+-------------+
|   JK-15370|     1|CA-2016-122581| 21/8/2016|25/8/2016|Standard Class|FUR-CH-10002961|       7|573.17|     0.3| 63.69|    Jay Kimmel|United States|
|   LB-16795|     3|US-2016-157490| 6/10/2016|7/10/2016|   First Class|OFF-BI-10002824|       4|  17.0|     0.7|-14.92|Laurel Beltran|United States|
+-----------+------+--------------+----------+---------+--------------+---------------+--------+------+--------+------+--------------+-------------+
only showing top 2 rows


In [15]:
enriched_orders = orders.join(
    enriched_products.select("Product ID", "Category", "Sub-Category"),
    on="Product ID", how="left")

In [16]:
enriched_orders.show(2)


+---------------+-----------+------+--------------+----------+---------+--------------+--------+------+--------+------+--------------+-------------+---------------+------------+
|     Product ID|Customer ID|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Quantity| Price|Discount|Profit| Customer Name|      Country|       Category|Sub-Category|
+---------------+-----------+------+--------------+----------+---------+--------------+--------+------+--------+------+--------------+-------------+---------------+------------+
|FUR-CH-10002961|   JK-15370|     1|CA-2016-122581| 21/8/2016|25/8/2016|Standard Class|       7|573.17|     0.3| 63.69|    Jay Kimmel|United States|      Furniture|      Chairs|
|OFF-BI-10002824|   LB-16795|     3|US-2016-157490| 6/10/2016|7/10/2016|   First Class|       4|  17.0|     0.7|-14.92|Laurel Beltran|United States|Office Supplies|     Binders|
+---------------+-----------+------+--------------+----------+---------+--------------+--------+------+-------

Agregate

In [17]:
orders_with_year = enriched_orders.withColumn(
    "Year", F.year(F.to_date("Order Date", "d/M/yyyy")))

In [18]:
orders_with_year.show(2)


+---------------+-----------+------+--------------+----------+---------+--------------+--------+------+--------+------+--------------+-------------+---------------+------------+----+
|     Product ID|Customer ID|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Quantity| Price|Discount|Profit| Customer Name|      Country|       Category|Sub-Category|Year|
+---------------+-----------+------+--------------+----------+---------+--------------+--------+------+--------+------+--------------+-------------+---------------+------------+----+
|FUR-CH-10002961|   JK-15370|     1|CA-2016-122581| 21/8/2016|25/8/2016|Standard Class|       7|573.17|     0.3| 63.69|    Jay Kimmel|United States|      Furniture|      Chairs|2016|
|OFF-BI-10002824|   LB-16795|     3|US-2016-157490| 6/10/2016|7/10/2016|   First Class|       4|  17.0|     0.7|-14.92|Laurel Beltran|United States|Office Supplies|     Binders|2016|
+---------------+-----------+------+--------------+----------+---------+-------------

In [19]:
orders_with_year = orders_with_year.fillna("Unknown", subset=["Category", "Sub-Category"])


In [20]:
profit_aggregate = (orders_with_year
    .groupBy("Year", "Category", "Sub-Category", "Customer ID", "Customer Name")
    .agg(F.round(F.sum("Profit"), 2).alias("Profit")))

In [21]:
profit_aggregate.show(5)

+----+---------------+------------+-----------+---------------+-------+
|Year|       Category|Sub-Category|Customer ID|  Customer Name| Profit|
+----+---------------+------------+-----------+---------------+-------+
|2016|      Furniture|      Tables|   VW-21775|Victoria Wilson|-231.41|
|2015|Office Supplies|      Labels|   VW-21775|Victoria Wilson|   8.88|
|2014|Office Supplies|   Fasteners|   VW-21775|Victoria Wilson|   2.68|
|2017|Office Supplies|     Storage|   VW-21775|Victoria Wilson| 500.44|
|2015|     Technology|      Phones|   VW-21775|Victoria Wilson|  -14.7|
+----+---------------+------------+-----------+---------------+-------+
only showing top 5 rows


In [22]:
profit_aggregate.createOrReplaceTempView("profit_aggregate")

SQL

In [23]:
spark.sql("""
    SELECT Year, ROUND(SUM(Profit), 2) AS Total_Profit
    FROM profit_aggregate
    GROUP BY Year
    ORDER BY Year
""").show()

+----+------------+
|Year|Total_Profit|
+----+------------+
|2014|    39185.75|
|2015|    63073.09|
|2016|    65073.23|
|2017|   111084.96|
+----+------------+



In [24]:
spark.sql("""
    SELECT Year, Category, ROUND(SUM(Profit), 2) AS Total_Profit
    FROM profit_aggregate
    GROUP BY Year, Category
    ORDER BY Year, Category
""").show(5)

+----+---------------+------------+
|Year|       Category|Total_Profit|
+----+---------------+------------+
|2014|      Furniture|    -5331.04|
|2014|Office Supplies|    22500.32|
|2014|     Technology|    21493.35|
|2014|        Unknown|      523.12|
|2015|      Furniture|     3027.18|
+----+---------------+------------+
only showing top 5 rows


In [25]:
spark.sql("""
    SELECT `Customer ID`, `Customer Name`, ROUND(SUM(Profit), 2) AS Total_Profit
    FROM profit_aggregate
    GROUP BY `Customer ID`, `Customer Name`
    ORDER BY Total_Profit DESC
""").show(4, truncate=False)

+-----------+-------------+------------+
|Customer ID|Customer Name|Total_Profit|
+-----------+-------------+------------+
|FH-14275   |Frank Hawley |13400.32    |
|TC-20980   |Tamara Chand |8981.32     |
|RB-19360   |Raymond Buch |6976.36     |
|SC-20095   |Sanjit Chand |5757.3      |
+-----------+-------------+------------+
only showing top 4 rows


In [26]:
spark.sql("""
    SELECT `Customer ID`, `Customer Name`, Year, ROUND(SUM(Profit), 2) AS Total_Profit
    FROM profit_aggregate
    GROUP BY `Customer ID`, `Customer Name`, Year
    ORDER BY `Customer ID`, Year
""").show(6, truncate=False)

+-----------+-------------+----+------------+
|Customer ID|Customer Name|Year|Total_Profit|
+-----------+-------------+----+------------+
|AA-10315   |Alex Avila   |2014|280.68      |
|AA-10315   |Alex Avila   |2015|7.0         |
|AA-10315   |Alex Avila   |2016|-747.09     |
|AA-10315   |Alex Avila   |2017|96.5        |
|AA-10375   |Allen Armold |2014|16.72       |
|AA-10375   |Allen Armold |2015|69.03       |
+-----------+-------------+----+------------+
only showing top 6 rows


PYTEST

In [ ]:
!pip -q install pytest

In [27]:
%%writefile steps.py
import pandas as pd
import glob, json, os
from pyspark.sql import SparkSession, functions as F

# Read raw files
def read_raw(spark, folder):
    customer_file = glob.glob(os.path.join(folder, "Customer*.xlsx"))[0]
    product_file = glob.glob(os.path.join(folder, "Products*.csv"))[0]
    order_file   = glob.glob(os.path.join(folder, "Orders*.json"))[0]

    customers = pd.read_excel(customer_file, dtype=str).fillna("")
    products  = pd.read_csv(product_file)
    with open(order_file) as f:
        orders = pd.DataFrame(json.load(f))

    return (spark.createDataFrame(customers),
            spark.createDataFrame(products),
            spark.createDataFrame(orders))

# Clean
def enrich_customers_products(customers, products):
    customers = customers.select("Customer ID", "Customer Name", "Country")
    products  = products.dropDuplicates(["Product ID"]).select("Product ID", "Category", "Sub-Category")
    return customers, products

# Join
def enrich_orders(orders, customers, products):
    return (orders
        .withColumn("Profit", F.round("Profit", 2))
        .join(customers, "Customer ID", "left")
        .join(products, "Product ID", "left"))

# Agg
def aggregate_profit(orders):
    return (orders
        .withColumn("Year", F.year(F.to_date("Order Date", "d/M/yyyy")))
        .fillna("Unknown", subset=["Category", "Sub-Category"])
        .groupBy("Year", "Category", "Sub-Category", "Customer ID")
        .agg(F.round(F.sum("Profit"), 2).alias("Profit")))


Writing steps.py


In [ ]:
# %%writefile test_steps.py
# import json, pandas as pd, pytest
# from pyspark.sql import SparkSession
# from steps import read_raw, make_enriched_tables, make_enriched_orders, make_aggregate

# @pytest.fixture(scope="session")
# def spark():
#     s = SparkSession.builder.master("local[1]").appName("tests").getOrCreate()
#     yield s
#     s.stop()

# def test_read_raw_files(spark, tmp_path):
#     pd.DataFrame({"Customer ID": ["C1"], "Customer Name": ["Ann"]}).to_excel(tmp_path / "Customer.xlsx", index=False)
#     pd.DataFrame({"Product ID": ["P1"], "Category": ["Furniture"]}).to_csv(tmp_path / "Products.csv", index=False)
#     with open(tmp_path / "Orders.json", "w") as f:
#         json.dump([{"Order ID": "O1", "Profit": 1.5}], f)

#     customers, products, orders = read_raw(spark, str(tmp_path))
#     assert customers.count() == 1
#     assert "Customer ID" in customers.columns
#     assert products.count() == 1
#     assert orders.count() == 1

# def test_enriched_tables_and_orders(spark):
#     customers = spark.createDataFrame([("C1","Ann","India")], ["Customer ID","Customer Name","Country"])
#     products  = spark.createDataFrame([("P1","Furniture","Chairs"),("P1","Furniture","Chairs")],
#                                       ["Product ID","Category","Sub-Category"])
#     orders    = spark.createDataFrame([("O1","1/1/2016","C1","P1",3.14159)],
#                                       ["Order ID","Order Date","Customer ID","Product ID","Profit"])

#     enriched_customers, enriched_products = make_enriched_tables(customers, products)
#     assert enriched_products.count() == 1   # duplicate removed

#     enriched_orders = make_enriched_orders(orders, enriched_customers, enriched_products).collect()[0]
#     assert enriched_orders["Profit"] == 3.14
#     assert enriched_orders["Customer Name"] == "Ann"

# def test_aggregate_profit(spark):
#     customers = spark.createDataFrame([("C1","Ann","India")], ["Customer ID","Customer Name","Country"])
#     products  = spark.createDataFrame([("P1","Furniture","Chairs")], ["Product ID","Category","Sub-Category"])
#     orders    = spark.createDataFrame([("O1","1/1/2016","C1","P1",10.0),("O2","2/1/2016","C1","P1",2.5)],
#                                       ["Order ID","Order Date","Customer ID","Product ID","Profit"])

#     agg = make_aggregate(make_enriched_orders(orders, customers, products))
#     total = sum(r["Profit"] for r in agg.collect())
#     assert total == pytest.approx(12.5)


In [28]:
%%writefile test_steps.py
import pytest
from pyspark.sql import SparkSession
from steps import read_raw, enrich_customers_products, enrich_orders, aggregate_profit


@pytest.fixture(scope="session")
def spark():
    s = SparkSession.builder.master("local[1]").appName("real-data-tests").getOrCreate()
    yield s
    s.stop()


def test_step1_read_raw_files(spark):
    folder = "/content"
    customers, products, orders = read_raw(spark, folder)
    assert customers.count() > 0
    assert products.count() > 0
    assert orders.count() > 0
    assert "Customer ID" in customers.columns
    assert "Product ID" in products.columns
    assert "Order ID" in orders.columns


def test_step2_enriched_tables(spark):
    folder = "/content"
    customers, products, _ = read_raw(spark, folder)
    enriched_customers, enriched_products = enrich_customers_products(customers, products)
    assert "Customer Name" in enriched_customers.columns
    assert "Country" in enriched_customers.columns
    assert "Category" in enriched_products.columns
    assert "Sub-Category" in enriched_products.columns
    assert enriched_products.count() <= products.count()


def test_step3_enriched_orders(spark):
    folder = "/content"
    customers, products, orders = read_raw(spark, folder)
    enriched_customers, enriched_products = enrich_customers_products(customers, products)
    enriched_orders = enrich_orders(orders, enriched_customers, enriched_products)
    assert "Profit" in enriched_orders.columns
    assert "Customer Name" in enriched_orders.columns
    assert "Category" in enriched_orders.columns
    assert enriched_orders.count() == orders.count()


def test_step4_aggregate_profit(spark):
    folder = "/content"
    customers, products, orders = read_raw(spark, folder)
    enriched_customers, enriched_products = enrich_customers_products(customers, products)
    enriched_orders = enrich_orders(orders, enriched_customers, enriched_products)
    agg = aggregate_profit(enriched_orders)
    assert "Year" in agg.columns
    assert "Profit" in agg.columns
    #sanity check
    total_orders = enriched_orders.groupBy().sum("Profit").collect()[0][0]
    total_agg = agg.groupBy().sum("Profit").collect()[0][0]
    assert total_orders == pytest.approx(total_agg)


def test_pipeline_integration(spark):
    folder = "/content"
    customers, products, orders = read_raw(spark, folder)
    enriched_customers, enriched_products = enrich_customers_products(customers, products)
    enriched_orders = enrich_orders(orders, enriched_customers, enriched_products)
    agg = aggregate_profit(enriched_orders)
    # pipeline sanity checks
    assert customers.count() > 0
    assert products.count() > 0
    assert orders.count() > 0
    assert enriched_orders.count() == orders.count()
    assert agg.count() > 0


Writing test_steps.py


In [29]:
!python -m pytest test_steps.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4, typeguard-4.6.0
collected 5 items                                                              

test_steps.py::test_step1_read_raw_files PASSED                          [ 20%]
test_steps.py::test_step2_enriched_tables PASSED                         [ 40%]
test_steps.py::test_step3_enriched_orders PASSED                         [ 60%]
test_steps.py::test_step4_aggregate_profit PASSED                        [ 80%]
test_steps.py::test_pipeline_integration PASSED                          [100%]

============================== 5 passed in 39.03s ==============================
